# Lab 3: AgentCore Code Interpreter를 활용한 해결 에이전트

## 개요
AgentCore Code Interpreter를 사용하여 인프라를 안전하게 자동화하고 스크립트를 실행하는 Strands 기반 해결 에이전트를 구축합니다.

## 목표
- AgentCore Code Interpreter가 통합된 Strands 에이전트 생성
- 해결 계획 생성 및 안전한 스크립트 실행을 위한 도구 구축
- 인프라 문제 해결 시나리오를 사용한 에이전트 테스트
- 액세스 및 실행 안전성 검증

## 학습 내용
- AgentCore Code Interpreter를 Strands 에이전트와 통합하는 방법
- 적절한 제어를 적용하여 안전한 문제 해결 워크플로를 구현하는 방법
- 인프라 자동화 도구를 생성하는 방법
- 에이전트의 문제 해결 계획 및 실행 패턴

## 아키텍처 개요
```
┌─────────────────┐    ┌──────────────────────┐    ┌─────────────────────┐
│   User Request  │───▶│  Strands Agent       │───▶│  AgentCore Code     │
│                 │    │  (Remediation)       │    │  Interpreter        │
└─────────────────┘    └──────────────────────┘    └─────────────────────┘
                                │                            │
                                ▼                            ▼
                       ┌──────────────────────┐    ┌─────────────────────┐
                       │  Remediation Tools   │    │  Secure Python      │
                       │  ├─ Plan Generation  │    │  Execution          │
                       │  ├─ Review   Gate    │    │  ├─ Session Mgmt    │
                       │  └─ Script Execution │    │  ├─ Code Streaming  │
                       └──────────────────────┘    │  └─ Error Handling  │
                                │                  └─────────────────────┘
                                ▼                            │
                       ┌──────────────────────┐              ▼
                       │  Infrastructure      │    ┌─────────────────────┐
                       │  Changes             │◀───│  Execution Results  │
                       │  (Approved Only)     │    │  & Validation       │
                       └──────────────────────┘    └─────────────────────┘
```

**핵심 구성 요소:**
- **2단계 프로세스**: 계획 → 검토 → 실행
- **안전한 실행**: AgentCore Code Interpreter가 격리된 환경 제공
- **위험 평가**: 각 단계에 대한 종합적인 영향 분석

## 0. 필수 패키지 설치

먼저 이 셀을 실행하여 모든 종속성이 설치되었는지 확인합니다.

In [ ]:
# %pip install -q -r requirements.txt
print("✅ Workshop dependencies installed")

## 1. 필수 모듈 가져오기

In [ ]:
### 1. 모듈 가져오기

# AWS SDK 및 구성
import json
import boto3
import logging
import uuid
from datetime import datetime
from typing import Dict

# Strands 프레임워크
from strands import Agent
from strands.models import BedrockModel
from strands.tools import tool

# Bedrock AgentCore Starter Toolkit
from bedrock_agentcore_starter_toolkit import Runtime
from lab_helpers.config import AWS_REGION, WORKSHOP_NAME
from lab_helpers.parameter_store import get_parameter, put_parameter
from lab_helpers.constants import PARAMETER_PATHS
from lab_helpers.lab_03.gateway_setup import AgentCoreGatewaySetup

# 워크숍 구성
from botocore.config import Config
from lab_helpers.config import MODEL_ID, AWS_PROFILE

# Lab-03 배포 헬퍼
from lab_helpers.lab_03 import (
    AgentCoreRuntimeDeployer,
    AgentCoreGatewaySetup,
)

# 파일 시스템 작업

# Notebook 로깅 구성
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

# Code Interpreter 전역 변수
agentcore_code_interpreter = None
CODE_INTERPRETER_AVAILABLE = False

print("✅ Imports loaded")
print(f"   Workshop: {WORKSHOP_NAME}")
print(f"   Region: {AWS_REGION}")
print(f"   Model: {MODEL_ID}")

## 2. 사전 요구 사항 설정 및 확인

In [ ]:
# 해결 계획을 저장할 S3 버킷 생성
from botocore.exceptions import ClientError
from lab_helpers.config import AWS_REGION

s3_client = boto3.client("s3", region_name=AWS_REGION)
unique_suffix = str(uuid.uuid4())[:8]
bucket_name = f"sre-workshop-remediation-plans-{unique_suffix}"

try:
    # us-east-1에서는 LocationConstraint를 사용하지 않음
    if AWS_REGION == "us-east-1":
        response = s3_client.create_bucket(Bucket=bucket_name)
    else:
        response = s3_client.create_bucket(
            Bucket=bucket_name,
            CreateBucketConfiguration={"LocationConstraint": AWS_REGION},
        )

    print(f"✅ Created bucket: {bucket_name}")

    # Parameter Store에 저장
    ssm = boto3.client("ssm", region_name=AWS_REGION)
    ssm.put_parameter(
        Name="/aiml301_sre_workshop/remediation_s3_bucket",
        Value=bucket_name,
        Type="String",
        Overwrite=True,
    )
    print("✅ Stored in Parameter Store")

except ClientError as e:
    print(f"❌ Error: {e}")


try:
    # AWS 자격 증명 테스트
    sts_client = boto3.client("sts", region_name=AWS_REGION)
    identity = sts_client.get_caller_identity()
    account_id = identity["Account"]

    # AgentCore Code Interpreter 가용성 테스트
    agentcore_test = boto3.client("bedrock-agentcore", region_name=AWS_REGION)

    print(f"✅ Prerequisites verified: AWS Account {account_id}, AgentCore Code Interpreter available")
    print(f"   Region: {AWS_REGION}")
    print(f"   Profile: {AWS_PROFILE}")
    print(f"   Model ID: {MODEL_ID}")
    print(f"   Identity: {identity.get('Arn', 'Unknown')}")

except Exception as e:
    print(f"❌ Error: {e}")
    print("Please ensure AWS credentials are configured and AgentCore Code Interpreter permissions are available.")

In [ ]:
# SSM Parameter Store에 저장
parameter_name = "/aiml301_sre_workshop/remediation_s3_bucket"
ssm = boto3.client("ssm", region_name="us-west-2")
parameter = ssm.get_parameter(Name=parameter_name)
retrieved_bucket_name = parameter["Parameter"]["Value"]

## 3. 사용자 지정 Code Interpreter 설정

**목표:** 사용자 지정 IAM 실행 역할을 사용하는 AgentCore Code Interpreter를 생성합니다.

**접근 방식:** 
1. 적절한 신뢰 정책과 권한을 갖춘 사용자 지정 IAM 실행 역할 생성
2. PUBLIC 네트워크 모드에서 사용자 지정 Code Interpreter 생성
3. 클라이언트 및 세션 관리 함수 초기화

**핵심 학습 내용:** 사용 사례에 맞는 특정 권한으로 사용자 지정 Code Interpreter를 생성하는 방법을 알아봅니다.

### 3.1 사용자 지정 Code Interpreter용 IAM 정책 및 역할 설정

In [ ]:
### 3.1: 사용자 지정 IAM 실행 역할 생성


def create_custom_code_interpreter_role():
    """사용자 지정 code interpreter용 IAM 실행 역할을 생성합니다."""
    iam_client = boto3.client("iam")
    sts_client = boto3.client("sts")
    account_id = sts_client.get_caller_identity()["Account"]

    role_name = f"{WORKSHOP_NAME}-CodeInterpreterRole"

    # 신뢰 정책 - bedrock-agentcore 서비스가 역할을 수임하도록 허용
    trust_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Sid": "AssumeRolePolicy",
                "Effect": "Allow",
                "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
                "Action": "sts:AssumeRole",
                "Condition": {
                    "StringEquals": {"aws:SourceAccount": account_id},
                    "ArnLike": {"aws:SourceArn": f"arn:aws:bedrock-agentcore:{AWS_REGION}:{account_id}:*"},
                },
            }
        ],
    }

    # Code Interpreter 실행을 위한 권한 정책
    with open("lab_helpers/lab_03/code_interpreter_permissions_policy.json", "r") as f:
        ci_permissions_policy = f.read()
        ci_permissions_policy = ci_permissions_policy.replace("{{ACCOUNT_ID}}", account_id)
        ci_permissions_policy = ci_permissions_policy.replace("{{REGION}}", AWS_REGION)

    try:
        # 역할 생성
        response = iam_client.create_role(
            RoleName=role_name,
            AssumeRolePolicyDocument=json.dumps(trust_policy),
            Description="Custom execution role for AgentCore Code Interpreter",
            Tags=[{"Key": "Workshop", "Value": WORKSHOP_NAME}],
        )
        role_arn = response["Role"]["Arn"]
        logger.info(f"✅ Created IAM role: {role_name}")
    except iam_client.exceptions.EntityAlreadyExistsException:
        response = iam_client.get_role(RoleName=role_name)
        role_arn = response["Role"]["Arn"]
        logger.info(f"✅ Using existing IAM role: {role_name}")

    # 권한 정책 연결
    try:
        iam_client.put_role_policy(
            RoleName=role_name,
            PolicyName="CodeInterpreterExecutionPolicy",
            PolicyDocument=ci_permissions_policy,
        )
        logger.info("✅ Attached permissions policy")
    except Exception as e:
        logger.warning(f"Policy may already exist: {e}")

    return role_arn, role_name


# 역할 생성
print("🔧 Creating custom IAM execution role...")
custom_role_arn, custom_role_name = create_custom_code_interpreter_role()
print(f"✅ IAM Role: {custom_role_name}")
print(f"   ARN: {custom_role_arn}")

### 3.2 AgentCore 사용자 지정 Code Interpreter 생성

In [ ]:
### 3.2: 사용자 지정 Code Interpreter 생성

import time


def create_custom_code_interpreter(role_arn):
    """실행 역할을 사용해 사용자 지정 code interpreter를 생성합니다."""
    agentcore_control = boto3.client("bedrock-agentcore-control", region_name=AWS_REGION)

    interpreter_name = f"{WORKSHOP_NAME}_custom_code_interpreter"

    try:
        response = agentcore_control.create_code_interpreter(
            name=interpreter_name,
            executionRoleArn=role_arn,
            networkConfiguration={"networkMode": "PUBLIC"},
            description="Custom code interpreter for remediation agent",
            clientToken=str(uuid.uuid4()),
        )
        interpreter_id = response["codeInterpreterId"]
        interpreter_arn = response["codeInterpreterArn"]
        status = response["status"]
        logger.info(f"✅ Created code interpreter: {interpreter_id}")

    except agentcore_control.exceptions.ConflictException:
        logger.info("⚠️  Code interpreter exists. Finding it...")
        # 실제 ID를 찾기 위한 목록 조회(키는 codeInterpreters가 아니라 codeInterpreterSummaries)
        list_response = agentcore_control.list_code_interpreters()
        all_items = list_response.get("codeInterpreterSummaries", [])

        interpreter_id = None
        for item in all_items:
            if item.get("name") == interpreter_name:
                interpreter_id = item["codeInterpreterId"]
                interpreter_arn = item["codeInterpreterArn"]
                status = item["status"]
                logger.info(f"✅ Found existing interpreter: {interpreter_id}")
                break

        if not interpreter_id:
            raise Exception(f"Interpreter '{interpreter_name}' exists but not found in list")

    # READY 상태가 될 때까지 대기
    if status == "CREATING":
        print("⏳ Waiting for code interpreter to be ready...")
        for _ in range(30):
            check_response = agentcore_control.get_code_interpreter(codeInterpreterIdentifier=interpreter_id)
            status = check_response["status"]
            if status == "READY":
                logger.info("✅ Code interpreter is READY")
                break
            elif status == "CREATE_FAILED":
                raise Exception("Code interpreter creation failed")
            time.sleep(10)

    return interpreter_id, interpreter_arn


# IAM 역할 전파 대기
print("⏳ Waiting for IAM role to propagate (10 seconds)...")
time.sleep(10)

# 사용자 지정 Code Interpreter 생성
print("\n🔧 Creating custom code interpreter...")
CUSTOM_INTERPRETER_ID, CUSTOM_INTERPRETER_ARN = create_custom_code_interpreter(custom_role_arn)
print("✅ Custom Code Interpreter Created")
print(f"   ID: {CUSTOM_INTERPRETER_ID}")
print(f"   ARN: {CUSTOM_INTERPRETER_ARN}")
print("   Network Mode: PUBLIC")
print(f"   Execution Role: {custom_role_name}")

# SSM Parameter Store에 저장

ssm = boto3.client("ssm", region_name=AWS_REGION)
ssm.put_parameter(
    Name=PARAMETER_PATHS["lab_03"]["code_interpreter_id"],
    Value=CUSTOM_INTERPRETER_ID,
    Type="String",
    Overwrite=True,
)
ssm.put_parameter(
    Name=PARAMETER_PATHS["lab_03"]["code_interpreter_arn"],
    Value=CUSTOM_INTERPRETER_ARN,
    Type="String",
    Overwrite=True,
)
ssm.put_parameter(
    Name=PARAMETER_PATHS["lab_03"]["code_interpreter_role_arn"],
    Value=custom_role_arn,
    Type="String",
    Overwrite=True,
)
print("✅ Stored in SSM Parameter Store")

In [ ]:
ssm.get_parameter(Name=f"/{WORKSHOP_NAME}/lab-03/code-interpreter-id")["Parameter"]["Value"]

## 3.3 Code Interpreter 클라이언트 함수 초기화 및 Code Interpreter 세션 테스트

In [ ]:
### 3.3: Code Interpreter 클라이언트 함수 초기화


def initialize_code_interpreter_client():
    """AgentCore Code Interpreter 클라이언트를 초기화합니다."""
    global agentcore_code_interpreter, CODE_INTERPRETER_AVAILABLE

    try:
        agentcore_code_interpreter = boto3.client("bedrock-agentcore", region_name=AWS_REGION)
        CODE_INTERPRETER_AVAILABLE = True
        logger.info("✅ AgentCore Code Interpreter client initialized")
        return True
    except Exception as e:
        CODE_INTERPRETER_AVAILABLE = False
        logger.warning(f"⚠️ AgentCore Code Interpreter not available: {e}")
        return False


def start_code_interpreter_session():
    """사용자 지정 interpreter로 Code Interpreter 세션을 시작합니다."""
    if not CODE_INTERPRETER_AVAILABLE:
        return None

    try:
        session_response = agentcore_code_interpreter.start_code_interpreter_session(
            codeInterpreterIdentifier=CUSTOM_INTERPRETER_ID,  # custom interpreter 사용
            name=f"remediation-session-{uuid.uuid4()}",
            sessionTimeoutSeconds=1800,  # 30분
        )

        session_id = session_response.get("sessionId")
        logger.info(f"✅ Code Interpreter session started: {session_id}")
        return session_id

    except Exception as e:
        logger.error(f"❌ Failed to start Code Interpreter session: {e}")
        return None


def stop_code_interpreter_session(session_id: str):
    """Code Interpreter 세션을 중지합니다."""
    if not session_id or not CODE_INTERPRETER_AVAILABLE:
        return

    try:
        agentcore_code_interpreter.stop_code_interpreter_session(
            codeInterpreterIdentifier=CUSTOM_INTERPRETER_ID,  # custom interpreter 사용
            sessionId=session_id,
        )
        logger.info(f"✅ Code Interpreter session stopped: {session_id}")
    except Exception as e:
        logger.error(f"❌ Failed to stop Code Interpreter session: {e}")


def execute_remediation_code(session_id: str, code: str) -> Dict:
    """사용자 지정 AgentCore Code Interpreter로 문제 해결 코드를 실행합니다."""
    if not session_id:
        return {"error": "No Code Interpreter session available"}

    try:
        logger.info(f"🔧 Executing remediation code: {code}")

        execute_response = agentcore_code_interpreter.invoke_code_interpreter(
            codeInterpreterIdentifier=CUSTOM_INTERPRETER_ID,  # custom interpreter 사용
            sessionId=session_id,
            name="executeCode",
            arguments={"language": "python", "code": code},
        )

        # 스트리밍 응답 처리
        output_text = ""
        execution_status = "success"

        for event in execute_response.get("stream", []):
            if "result" in event:
                result = event["result"]
                if "content" in result:
                    for content_item in result["content"]:
                        if content_item.get("type") == "text":
                            output_text += content_item.get("text", "")
                        elif content_item.get("type") == "error":
                            execution_status = "error"
                            output_text += f"ERROR: {content_item.get('text', '')}"

        return {
            "execution_status": execution_status,
            "output": output_text,
            "session_id": session_id,
        }

    except Exception as e:
        logger.error(f"❌ Failed to execute remediation code: {e}")
        return {"error": f"Code execution failed: {str(e)}"}


# 초기화 테스트
if initialize_code_interpreter_client():
    print("\n✅ Custom Code Interpreter integration ready")
    print(f"   Interpreter ID: {CUSTOM_INTERPRETER_ID}")
    print("   Network Mode: PUBLIC")
    print(f"   Execution Role: {custom_role_name}")
    print(f"   Client Status: {CODE_INTERPRETER_AVAILABLE}")
    print("   Functions: initialize, start_session, stop_session, execute_code")

    # 세션 생성 테스트
    print("\n🧪 Testing session creation...")
    test_session_id = start_code_interpreter_session()
    if test_session_id:
        print(f"✅ Test session created: {test_session_id}")
        stop_code_interpreter_session(test_session_id)
        print("✅ Test session stopped")
    else:
        print("❌ Test session creation failed")
else:
    print("❌ Code Interpreter client initialization failed")
    print("   Check AWS credentials and AgentCore permissions")

## 3.4 관찰성 활성화
AgentCore에서 관찰성을 활성화하려면 먼저 Transaction Search를 활성화해야 합니다. 리전별로 한 번만 설정하면 됩니다.
Transaction Search는 다음 기능을 제공합니다.
상세 분석을 위한 구조화된 로그 형태의 스팬 수집, 세션 추적을 위한 X-Ray 트레이스 인덱싱, 모든 AgentCore Runtime에 대한 심층 트레이스 분석
자신의 계정에서 Transaction Search를 활성화하는 방법에 대한 자세한 내용은 AWS 문서를 참조하세요.

In [ ]:
# 1. 세션 설정
session = boto3.Session()
region = AWS_REGION
sts = session.client("sts")
account_id = sts.get_caller_identity()["Account"]
logs_client = session.client("logs")
xray_client = session.client("xray")


print(f"Configuring AgentCore Observability for Account: {account_id} in Region: {AWS_REGION}")

# ---------------------------------------------------------
# 1단계: 리소스 정책(멱등성에 가까움)
# ---------------------------------------------------------
policy_document = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "TransactionSearchXRayAccess",
            "Effect": "Allow",
            "Principal": {"Service": "xray.amazonaws.com"},
            "Action": "logs:PutLogEvents",
            "Resource": [
                f"arn:aws:logs:{region}:{account_id}:log-group:/aws/vendedlogs/xray:*",
                f"arn:aws:logs:{region}:{account_id}:log-group:aws/spans:*",
                f"arn:aws:logs:{region}:{account_id}:log-group:/aws/application-signals/*",
            ],
        }
    ],
}

try:
    logs_client.put_resource_policy(
        policyName="BedrockAgentCoreXRayPolicy",
        policyDocument=json.dumps(policy_document),
    )
    print("✅ Resource Policy created/updated successfully.")
except Exception as e:
    print(f"ℹ️ Note on Policy: {e}")

# ---------------------------------------------------------
# 2단계: 트레이스 대상 업데이트("이미 설정됨" 처리)
# ---------------------------------------------------------
try:
    xray_client.update_trace_segment_destination(Destination="CloudWatchLogs")
    print("✅ Trace segment destination set to CloudWatchLogs.")
except xray_client.exceptions.InvalidRequestException as e:
    # 이미 활성화되어 발생한 오류인지 확인
    if "already set" in str(e):
        print("✅ Trace destination was already set to CloudWatchLogs (Skipped).")
    else:
        print(f"❌ Error updating trace destination: {e}")

# ---------------------------------------------------------
# 3단계: 인덱싱 규칙 업데이트(수정됨: 문자열이 아닌 딕셔너리 전달)
# ---------------------------------------------------------
try:
    xray_client.update_indexing_rule(
        Name="Default",
        # 수정: json.dumps() 문자열이 아닌 Python 딕셔너리 전달
        Rule={"Probabilistic": {"DesiredSamplingPercentage": 5.0}},
    )
    print("✅ Indexing rule updated successfully.")
except Exception as e:
    print(f"❌ Error updating indexing rule: {e}")

In [ ]:
# 로그 그룹, 로그 소스 및 대상 생성 후 연결


def enable_observability_for_resource(resource_arn, resource_id, account_id, region=AWS_REGION):
    """
    Bedrock AgentCore 리소스(예: Memory Store)에 observability를 활성화합니다.
    """
    logs_client = boto3.client("logs", region_name=region)

    # 0단계: vended log 전송을 위한 새 로그 그룹 생성
    log_group_name = f"/aws/vendedlogs/bedrock-agentcore/{resource_id}"
    try:
        logs_client.create_log_group(logGroupName=log_group_name)
    except logs_client.exceptions.ResourceAlreadyExistsException:
        pass
    log_group_arn = f"arn:aws:logs:{region}:{account_id}:log-group:{log_group_name}"
    print(f"Resource CloudWatch Log Group: f{log_group_arn}")

    unique_suffix = str(uuid.uuid4())[:8]

    # 1단계: 로그 전송 소스 생성
    logs_source_response = logs_client.put_delivery_source(
        name=f"aiml301_custom_code_interpreter-{unique_suffix}-logs-source",
        logType="APPLICATION_LOGS",
        resourceArn=resource_arn,
    )

    # 2단계: 트레이스 전송 소스 생성
    traces_source_response = logs_client.put_delivery_source(
        name=f"aiml301_custom_code_interpreter-{unique_suffix}-traces-source",
        logType="TRACES",
        resourceArn=resource_arn,
    )

    # 3단계: 전송 대상 생성
    logs_destination_response = logs_client.put_delivery_destination(
        name=f"aiml301_custom_code_interpreter-{unique_suffix}-logs-dest",
        deliveryDestinationType="CWL",
        deliveryDestinationConfiguration={
            "destinationResourceArn": log_group_arn,
        },
    )

    # 트레이스에 필요
    traces_destination_response = logs_client.put_delivery_destination(
        name=f"aiml301_custom_code_interpreter-{unique_suffix}-traces-dest",
        deliveryDestinationType="XRAY",
    )

    # 4단계: 전송 생성(소스를 대상에 연결)
    logs_client.create_delivery(
        deliverySourceName=logs_source_response["deliverySource"]["name"],
        deliveryDestinationArn=logs_destination_response["deliveryDestination"]["arn"],
    )

    # 트레이스에 필요
    logs_client.create_delivery(
        deliverySourceName=traces_source_response["deliverySource"]["name"],
        deliveryDestinationArn=traces_destination_response["deliveryDestination"]["arn"],
    )

    print(f"Observability enabled for {resource_id}")


# Code Interpreter 리소스 ID 및 ARN 가져오기
resource_id = ssm.get_parameter(Name=f"/{WORKSHOP_NAME}/lab-03/code-interpreter-id")["Parameter"]["Value"]
resource_arn = f"arn:aws:bedrock-agentcore:{AWS_REGION}:{account_id}:code-interpreter-custom/{resource_id}"

delivery_ids = enable_observability_for_resource(resource_arn, resource_id, account_id)

## 4. Strands 에이전트 도구 생성

**목표:** 승인 워크플로를 포함한 문제 해결 계획 및 실행용 Strands 도구를 정의합니다.

**접근 방식:** 계획 생성, 실행 및 검증을 위한 @tool 데코레이터 함수를 생성합니다.

**핵심 학습 내용:** 필수 승인 단계를 포함한 안전한 문제 해결 워크플로를 구현하는 방법을 알아봅니다.

In [ ]:
### 4.1: 문제 해결 도구 정의 [execute_remediation_step]


@tool
def execute_remediation_step(remediation_code: str) -> str:
    """Execute remediation steps"""

    if not initialize_code_interpreter_client():
        return "AgentCore Code Interpreter not available"

    session_id = start_code_interpreter_session()
    if not session_id:
        return "Failed to start code interpreter session"

    try:
        execution_result = execute_remediation_code(session_id, remediation_code)

        if "error" in execution_result:
            return f"❌ failed: {execution_result['error']}"

        response = "# ✅ APPROVED EXECUTION - Results\n\n"
        response += "## Execution Output\n\n```\n"
        response += execution_result["output"]
        response += "\n```\n"

        return response

    except Exception as e:
        logger.error(f"❌ Error : {e}")
        return f"❌ remediation plan execution failed: {str(e)}"
    finally:
        stop_code_interpreter_session(session_id)

In [ ]:
### 4.2: 문제 해결 도구 정의 [validate_remediation_environment]


@tool
def validate_remediation_environment() -> str:
    """Validate that the remediation environment is ready"""

    logger.info("🔍 Validating remediation environment...")

    validation_results = {
        "code_interpreter_available": False,
        "session_creation": False,
        "aws_access": False,
        "environment_ready": False,
    }

    try:
        # Code Interpreter 초기화 테스트
        if initialize_code_interpreter_client():
            validation_results["code_interpreter_available"] = True

            # 세션 생성 테스트
            session_id = start_code_interpreter_session()
            if session_id:
                validation_results["session_creation"] = True
                validation_results["aws_access"] = True  # 데모를 위해 단순화
                stop_code_interpreter_session(session_id)

        validation_results["environment_ready"] = all(
            [
                validation_results["code_interpreter_available"],
                validation_results["session_creation"],
                validation_results["aws_access"],
            ]
        )

    except Exception as e:
        logger.error(f"❌ Environment validation failed: {e}")

    # 응답 형식 지정
    response = "# Remediation Environment Validation\n\n"
    response += f"**Validation Date**: {datetime.utcnow().isoformat()}\n\n"

    for check, status in validation_results.items():
        status_icon = "✅" if status else "❌"
        check_name = check.replace("_", " ").title()
        response += f"- **{check_name}**: {status_icon} {'PASS' if status else 'FAIL'}\n"

    if validation_results["environment_ready"]:
        response += "\n🎉 **Environment is READY for remediation**\n"
    else:
        response += "\n⚠️ **Environment is NOT READY**\n"

    return response

In [ ]:
### 4.3: 문제 해결 도구 정의 [persist_remediation_scripts_to_s3]
@tool
def persist_remediation_scripts_to_s3(file_key: str, content: str) -> dict:
    """Write a python scripts to S3 bucket.

    Args:
        bucket_name: Name of the S3 bucket
        file_key: The S3 key (path/filename) where the file will be stored
        content: The content to write to the file
        region: AWS region (default: us-west-2)
        content_type: MIME type of the content (default: text/plain)
    """
    bucket_name = retrieved_bucket_name
    region = AWS_REGION
    try:
        s3_client = boto3.client("s3", region_name=region)

        # S3에 쓰기
        s3_client.put_object(Bucket=bucket_name, Key=file_key, Body=content.encode("utf-8"))

        # S3 URL 생성
        s3_url = f"s3://{bucket_name}/{file_key}"
        https_url = f"https://{bucket_name}.s3.{region}.amazonaws.com/{file_key}"

        result = {
            "success": True,
            "message": "Successfully wrote file to S3",
            "bucket": bucket_name,
            "key": file_key,
            "s3_url": s3_url,
            "https_url": https_url,
            "size_bytes": len(content.encode("utf-8")),
        }

        return {
            "status": "success",
            "content": [{"text": f"✓ File written  to {s3_url}"}, {"json": result}],
        }

    except Exception as e:
        error_msg = f"Failed to write file to S3: {str(e)}"
        return {"status": "error", "content": [{"text": error_msg}]}

In [ ]:
### 4.4: 문제 해결 도구 정의 [read_remediation_scripts_from_s3]
@tool
def read_remediation_scripts_from_s3(prefix: str = "") -> dict:
    """Read all files from an S3 bucket and return their contents.

    Args:
        prefix: Optional prefix to filter files (e.g., 'crm-remediation')
    """
    bucket_name = retrieved_bucket_name
    region = AWS_REGION
    max_files = 100

    try:
        s3_client = boto3.client("s3", region_name=region)

        # 객체 목록 조회
        list_params = {"Bucket": bucket_name, "MaxKeys": max_files}
        if prefix:
            list_params["Prefix"] = prefix

        response = s3_client.list_objects_v2(**list_params)

        if "Contents" not in response:
            return {
                "status": "success",
                "content": [
                    {"text": f"No files found in s3://{bucket_name}/{prefix}"},
                    {
                        "json": {
                            "success": True,
                            "bucket": bucket_name,
                            "prefix": prefix,
                            "file_count": 0,
                            "files": [],
                        }
                    },
                ],
            }

        files_data = []
        total_size = 0

        # 각 파일 읽기
        for obj in response["Contents"]:
            file_key = obj["Key"]

            # 디렉터리 건너뛰기(/로 끝나는 키)
            if file_key.endswith("/"):
                continue

            try:
                # 파일 내용 읽기
                file_response = s3_client.get_object(Bucket=bucket_name, Key=file_key)
                content = file_response["Body"].read().decode("utf-8")

                file_info = {
                    "key": file_key,
                    "s3_url": f"s3://{bucket_name}/{file_key}",
                    "size": obj["Size"],
                    "last_modified": obj["LastModified"].isoformat(),
                    "content": content,
                }
                files_data.append(file_info)
                total_size += obj["Size"]

            except Exception as file_error:
                # 파일을 읽을 수 없으면 오류 정보를 포함하고 계속 진행
                files_data.append(
                    {
                        "key": file_key,
                        "s3_url": f"s3://{bucket_name}/{file_key}",
                        "size": obj["Size"],
                        "last_modified": obj["LastModified"].isoformat(),
                        "error": str(file_error),
                    }
                )

        result = {
            "success": True,
            "message": f"Successfully read {len(files_data)} files from S3",
            "bucket": bucket_name,
            "prefix": prefix,
            "file_count": len(files_data),
            "total_size_bytes": total_size,
            "files": files_data,
        }

        return {
            "status": "success",
            "content": [
                {"text": f"✓ Read {len(files_data)} files from s3://{bucket_name}/{prefix}"},
                {"json": result},
            ],
        }

    except Exception as e:
        error_msg = f"Failed to read files from S3: {str(e)}"
        return {"status": "error", "content": [{"text": error_msg}]}

In [ ]:
# 현재 애플리케이션 아키텍처 및 세부 정보 가져오기
with open("lab_helpers/lab_03/app_arch.txt", "r") as f:
    current_app_architecture = f.read()

## 5. Strands 에이전트 생성

**목표:** 문제 해결 도구와 적절한 시스템 프롬프트를 사용하는 Strands 에이전트를 생성합니다.

**접근 방식:** Bedrock 모델과 문제 해결 도구를 사용하여 에이전트를 구성합니다.

**핵심 학습 내용:** 인프라 문제 해결 워크플로용 에이전트를 구성하는 방법을 알아봅니다.

In [ ]:
### 5.1: 문제 해결 도구를 사용하는 Strands 에이전트 생성


def setup_agent(region=AWS_REGION):
    """문제 해결 도구로 Strands 에이전트를 설정합니다."""
    try:
        if not initialize_code_interpreter_client():
            logger.error("❌ Failed to initialize code interpreter client")
            return None

        BOTO3_CONFIG = Config(
            read_timeout=300,  # model 응답에 5분
            connect_timeout=60,  # 연결에 1분
            retries={"max_attempts": 3, "mode": "adaptive"},
        )
        model = BedrockModel(
            model_id=MODEL_ID,
            streaming=True,
            max_tokens=4000,
            boto_client_config=BOTO3_CONFIG,
        )
        system_prompt = f"""
            You are an AWS application remediation agent that helps in creating remediation plans in markdown format (no code execution). 
            Here are the application details and architecture: {current_app_architecture}
            Think step and step, break down the problems into smaller steps and use the persist_remediation_scripts_to_s3 tool to persist remediation plans.

FOCUS: Generate plans to restore system availability. No long-term improvements.

After the remediation plan has been created and persisted, provide the below summary:
1. **Issue Summary** - Brief description of the problem
2. **Root Cause** - Identified cause based on diagnostics
3. **Remediation Plan** - High level summary of the proposed fixes (numbered list)

        """
        agent = Agent(
            system_prompt=system_prompt,
            model=model,
            tools=[
                execute_remediation_step,
                validate_remediation_environment,
                persist_remediation_scripts_to_s3,
                read_remediation_scripts_from_s3,
            ],
        )

        logger.info("✅ SRE Remediation Agent ready with code interpreter tools")
        logger.info(f"🌍 Region: {region}")
        logger.info(f"🔧 Code interpreter integration: {CODE_INTERPRETER_AVAILABLE}")

        return agent

    except Exception as e:
        logger.error(f"❌ Failed to setup agent: {e}")
        return None


# 에이전트 설정
agent = setup_agent()
if agent:
    print("✅ Strands agent created successfully")
    print(f"   Model: {MODEL_ID}")
    print(
        "   Tools: 4 (execute_remediation_step, validate_remediation_environment, persist_remediation_scripts_to_s3, read_remediation_scripts_from_s3)"
    )
    print(f"   Code Interpreter: {CODE_INTERPRETER_AVAILABLE}")
else:
    print("❌ Agent setup failed!")
    print("   Check Code Interpreter initialization and AWS credentials")

## 진단 정보로 Memory의 컨텍스트 보강

선별된 Memory에서 추가 정보를 가져와 진단 정보로 컨텍스트를 보강해 보겠습니다.

In [ ]:
agent_memory_client = boto3.client("bedrock-agentcore", region_name=AWS_REGION)

memory_id = get_parameter(PARAMETER_PATHS["memory"]["memory_id"])
memory_session_id = get_parameter(PARAMETER_PATHS["memory"]["default_session_id"])

print(memory_id)
print(memory_session_id)
actor_id = "diagnostics_agent"


# 쓰기 성공 여부를 확인하기 위해 에이전트 Memory에 추가된 이벤트 나열
params = {
    "memoryId": memory_id,
    "actorId": actor_id,
    "sessionId": memory_session_id,
    "includePayloads": True,
}
# 모든 메시지 가져오기
response = agent_memory_client.list_events(**params)
additional_context = ""
for event in response.get("events", []):
    payload = event.get("payload", [])
    for i, item in enumerate(payload):
        if "conversational" in item:
            text = item["conversational"]["content"]["text"]
            additional_context += text
additional_context

## 6. 문제 해결 워크플로 테스트

**목표:** 승인 단계를 포함한 전체 문제 해결 워크플로를 시연합니다.

**접근 방식:** 2단계 승인 프로세스로 인프라 문제 해결 분석을 실행합니다.

**핵심 학습 내용:** 계획부터 실행 승인까지 이어지는 엔드 투 엔드 문제 해결 프로세스를 알아봅니다.

In [ ]:
### 6.1: 전체 문제 해결 워크플로 실행


if agent:
    print("🚀 Starting Complete Remediation Workflow...")
    print("=" * 60)
    print()

    # 문제 해결 prompt 예시
    # remediation_prompt = f"""I need help with infrastructure remediation for our CRM application. We're experiencing: {additional_context} """

    remediation_prompt = f"""
    Help me fix  the dynamo DB throttling issues based on this diagnostic information: {additional_context}.
    
    """

    try:
        start_time = datetime.now()
        response = agent(remediation_prompt)
        analysis_time = (datetime.now() - start_time).total_seconds()

        print("\n🎯 REMEDIATION ANALYSIS RESULTS:")
        print(f"Analysis Time: {analysis_time:.2f} seconds")

        # 응답 표시
        response_content = response.message.get("content", [])
        if response_content:
            for content in response_content:
                if isinstance(content, dict) and "text" in content:
                    text = content["text"]
                    if len(text) > 2000:
                        print(f"\n📋 AGENT ANALYSIS (first 2000 chars):\n{text[:2000]}...")
                    else:
                        print(f"\n📋 AGENT ANALYSIS:\n{text}")
    except Exception as e:
        print(f"❌ Error: {e}")

    print("⚠️  Note: This will demonstrate the complete remediation planning workflow")

else:
    print("❌ Agent not available for workflow demonstration!")

## 7. AgentCore Runtime에 배포

**목표:** 서버리스 실행을 위해 문제 해결 에이전트를 Amazon Bedrock AgentCore Runtime에 배포합니다.

**접근 방식:** AgentCore와 호환되도록 에이전트를 변환하고 CLI를 사용하여 배포합니다.

**핵심 학습 내용:** Code Interpreter가 통합된 Strands 에이전트를 프로덕션 지원 서버리스 인프라에 배포하는 방법을 알아봅니다.

### 7.1: AgentCore 호환 에이전트 생성

먼저 필수 래퍼와 진입점을 포함하여 문제 해결 에이전트의 AgentCore 호환 버전을 생성해야 합니다.

In [ ]:
### 7.1: 사용자 지정 Runtime IAM 역할 생성

# 배포 도구 초기화
deployer = AgentCoreRuntimeDeployer(region=AWS_REGION, prefix=WORKSHOP_NAME, verbose=False)

# AgentCore Starter Toolkit 사전 요구 사항 확인
if not deployer.check_prerequisites():
    raise RuntimeError("Prerequisites not met. Install: pip install bedrock-agentcore-starter-toolkit")

# 사용자 지정 정책을 로드하고 자리 표시자 교체
iam = boto3.client("iam")
sts = boto3.client("sts")
account_id = sts.get_caller_identity()["Account"]
role_name = f"{WORKSHOP_NAME}_CustomRemediationRuntimeRole"

# 신뢰 정책 로드
with open("lab_helpers/lab_03/custom_runtime_trust_policy.json", "r") as f:
    trust_policy = f.read()
    trust_policy = trust_policy.replace("{{ACCOUNT_ID}}", account_id)
    trust_policy = trust_policy.replace("{{REGION}}", AWS_REGION)

# 권한 정책 로드
with open("lab_helpers/lab_03/custom_runtime_permissions.json", "r") as f:
    permissions_policy = f.read()
    permissions_policy = permissions_policy.replace("{{ACCOUNT_ID}}", account_id)
    permissions_policy = permissions_policy.replace("{{REGION}}", AWS_REGION)
    permissions_policy = permissions_policy.replace("{{PREFIX}}", WORKSHOP_NAME)

# 역할 생성 또는 업데이트
try:
    role = iam.get_role(RoleName=role_name)
    role_arn = role["Role"]["Arn"]
    iam.update_assume_role_policy(RoleName=role_name, PolicyDocument=trust_policy)
    print(f"✅ Using existing role: {role_name}")
except iam.exceptions.NoSuchEntityException:
    role = iam.create_role(
        RoleName=role_name,
        AssumeRolePolicyDocument=trust_policy,
        Description="Custom execution role for AgentCore Runtime",
    )
    role_arn = role["Role"]["Arn"]
    print(f"✅ Created custom role: {role_name}")
    import time

    time.sleep(10)

# 권한 연결
iam.put_role_policy(
    RoleName=role_name,
    PolicyName=f"{WORKSHOP_NAME}_RuntimePermissions",
    PolicyDocument=permissions_policy,
)

role_info = {"role_arn": role_arn, "role_name": role_name}
print("✅ Custom permissions attached")
print(f"   Role ARN: {role_arn}")

In [ ]:
### 7.1: 배포용 에이전트 코드

# 헬퍼 파일에서 에이전트 코드 로드
with open("lab_helpers/lab_03/runtime_mcp_agent_code.py", "r") as f:
    agentcore_agent_code = f.read()

print("✅ Agent code loaded from: lab_helpers/lab_03/runtime_mcp_agent_code.py")
print(f"   Code length: {len(agentcore_agent_code)} characters")
print("   Transport: streamable-http (AgentCore Runtime compatible)")

In [ ]:
### 7.1: 에이전트 코드를 디스크에 쓰기
with open("agent-remediation.py", "w") as f:
    f.write(agentcore_agent_code)

print("✅ Agent code written: agent-remediation.py")

### 7.2 JWT 권한 부여자를 사용하도록 Runtime 구성

**runtime.configure()의 기능:**
- 에이전트 코드 및 종속성 검증
- Dockerfile 및 AWS 구성 파일 생성
- 배포 청사진 준비(로컬 작업이며 아직 AWS 리소스는 생성되지 않음)
- 실행 역할 및 토큰 검증 설정

**JWT 권한 부여자 구성:**
`authorizer_configuration` 파라미터는 수신 토큰을 검증하는 방법을 Runtime에 지정합니다.
- **discoveryUrl**: Runtime이 서명 검증용 공개 키를 가져오는 Cognito OIDC 엔드포인트
- **allowedClients**: User Auth 클라이언트(직접 사용자)와 M2M 클라이언트(Gateway)를 모두 허용

**Runtime 자동 동작:**
```
Bearer token in request → Validate signature → Check issuer (Cognito) → Verify client ID in allowedClients → Allow or Reject
```

In [ ]:
from lab_helpers.parameter_store import get_parameter
from lab_helpers.constants import PARAMETER_PATHS

# Lab-01 SSM Parameter Store에서 Cognito 구성 가져오기
user_pool_id = get_parameter(PARAMETER_PATHS["cognito"]["user_pool_id"])
m2m_client_id = get_parameter(PARAMETER_PATHS["cognito"]["m2m_client_id"])
user_auth_client_id = get_parameter(PARAMETER_PATHS["cognito"]["user_auth_client_id"])

# Cognito discovery URL 구성 - Runtime은 이 URL에서 토큰 검증용 공개 키를 가져옴
discovery_url = f"https://cognito-idp.{AWS_REGION}.amazonaws.com/{user_pool_id}/.well-known/openid-configuration"

print("✅ Cognito configuration retrieved")
print(f"   Discovery URL: {discovery_url}")
print("   Allowed Clients: User Auth + M2M")
print(f"   User Auth Client ID: {user_auth_client_id}")
print(f"   M2M Client ID: {m2m_client_id}")

In [ ]:
### 7.2 b: JWT 권한 부여자를 사용하도록 Runtime 구성

# Runtime 객체 초기화
runtime = Runtime()

# JWT 권한 부여자 구성
# - discoveryUrl: Cognito OIDC 엔드포인트(Runtime이 공개 키를 자동으로 가져옴)
# - allowedClients: User Auth 및 M2M 클라이언트 모두 Runtime 호출 가능
authorizer_config = {
    "customJWTAuthorizer": {
        "discoveryUrl": discovery_url,
        "allowedClients": [user_auth_client_id, m2m_client_id],
    }
}

# 중요: JWT 토큰 검증을 사용하도록 Runtime 구성
print(f"\n🔍 Using execution role: {role_info['role_arn']}")
print(f"   Role name: {role_info['role_name']}")

runtime.configure(
    entrypoint="agent-remediation.py",
    execution_role=role_info["role_arn"],
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=AWS_REGION,
    agent_name=f"{WORKSHOP_NAME}_remediation_runtime",
    protocol="MCP",
    authorizer_configuration=authorizer_config,  # ← token validation 활성화
)

print("✅ Runtime configured with JWT authorizer")
print("   Protocol: MCP")
print("   JWT Token Validation: ENABLED")
print("   Allowed Tokens: User Auth + M2M")

### 7.3: Runtime을 AgentCore로 시작

Python SDK `runtime.launch()`를 사용하여 구성된 Runtime을 AgentCore에 배포합니다. 이 핵심 단계에서는 로컬 에이전트를 프로덕션 서버리스 서비스로 변환합니다.

#### 7.3 a: 시작 프로세스 이해

  **runtime.launch() 실행 중 수행되는 작업:**

  1️⃣ CodeBuild가 Docker 컨테이너 빌드를 시작합니다.

  2️⃣ requirements.txt의 종속성이 설치됩니다.

  3️⃣ 이미지가 Amazon ECR에 푸시됩니다(자동 생성).

  4️⃣ AgentCore가 Runtime을 MCP 서비스로 등록합니다.
  
  5️⃣ CloudWatch 로깅이 구성됩니다.

  ⏱️ **일반적인 소요 시간:** 5~10분

In [ ]:
# 모든 셀 실행 시나리오를 처리하기 위해 추가
import time

time.sleep(10)

In [ ]:
### 7.3 b: runtime.launch() 실행

print("\n🚀 Launching Runtime to AgentCore...\n")

try:
    # 충돌 시 자동 업데이트를 사용하여 Runtime 시작
    # 동기 방식으로 실행되며 CodeBuild 및 초기 상태 확인이 완료될 때까지 대기
    launch_result = runtime.launch(auto_update_on_conflict=True)

    # 배포 ARN 추출
    runtime_arn = launch_result.agent_arn

    print("✅ Runtime launched successfully!")
    print(f"   Runtime ARN: {runtime_arn}")
    print("\n📝 Configuration stored for next sections:")

except Exception as e:
    print(f"❌ Launch failed: {e}")
    print("\nTroubleshooting:")
    print("  • Check CodeBuild service limits")
    print("  • Verify ECR permissions in IAM role")
    print("  • Review CloudWatch logs for build errors")
    print("  • Ensure all dependencies in requirements.txt are correct")
    raise

In [ ]:
### 7.3c: Runtime 구성 저장

# Runtime 구성 추출 및 저장
runtime_arn = launch_result.agent_arn
runtime_id = getattr(launch_result, "agent_id", None)

# 구성 저장
from lab_helpers.lab_03 import store_runtime_configuration

store_runtime_configuration(runtime_arn, runtime_id, region=AWS_REGION, prefix=WORKSHOP_NAME)

print("\n✅ Runtime deployed and configured")
print(f"   ARN: {runtime_arn}")
print("   Ready for Gateway registration")

In [ ]:
# 로깅 구성 헬퍼 가져오기
from lab_helpers.lab_03.configure_logging import configure_runtime_logging

# Runtime용 CloudWatch Logs Delivery 구성
logging_config = configure_runtime_logging(
    runtime_arn=runtime_arn,  # 이전 cell에서 가져옴
    runtime_id=runtime_id,  # 이전 cell에서 가져옴
    region=AWS_REGION,
    log_type="APPLICATION_LOGS",  # 컨테이너 stdout/stderr log
)

print("\n📊 Logging Configuration Summary:")
print(f"  Log Group: {logging_config['log_group_name']}")
print(f"  Delivery Status: {logging_config['delivery_status']}")
print(f"  Delivery ID: {logging_config['delivery_id']}")

## 8. Cognito JWT 인증을 사용하는 Gateway 배포

**목표:** 수신 사용자 인증을 위해 Cognito JWT 권한 부여자를 사용하는 AgentCore Gateway를 배포합니다.

**접근 방식:** Gateway 액세스 제어에 Cognito User Auth Client 및 JWT 검증을 사용합니다.

**핵심 학습 내용:** Cognito 인증을 AgentCore Gateway에 통합하는 방법을 알아봅니다.

In [ ]:
# Lab-01에서 Cognito 자격 증명 가져오기
user_auth_client_id = get_parameter(PARAMETER_PATHS["cognito"]["user_auth_client_id"])
user_pool_id = get_parameter(PARAMETER_PATHS["cognito"]["user_pool_id"])

# Cognito OIDC discovery URL 구성
discovery_url = f"https://cognito-idp.{AWS_REGION}.amazonaws.com/{user_pool_id}/.well-known/openid-configuration"

# 헬퍼를 사용하여 Gateway IAM 역할 생성
gateway_setup = AgentCoreGatewaySetup(region=AWS_REGION, prefix=WORKSHOP_NAME, verbose=False)
role_info = gateway_setup.create_gateway_service_role()
role_arn = role_info["role_arn"]

# boto3로 Gateway 직접 생성(간단한 API 호출)
agentcore = boto3.client("bedrock-agentcore-control", region_name=AWS_REGION)

gateway_response = agentcore.create_gateway(
    name="aiml301-remediation-gateway",
    roleArn=role_arn,
    protocolType="MCP",
    authorizerType="CUSTOM_JWT",
    authorizerConfiguration={
        "customJWTAuthorizer": {
            "discoveryUrl": discovery_url,
            "allowedClients": [user_auth_client_id],
        }
    },
)

gateway_id = gateway_response["gatewayId"]
gateway_url = gateway_response["gatewayUrl"]

# 구성 저장
put_parameter(PARAMETER_PATHS["lab_03"]["gateway_id"], gateway_id)
put_parameter(PARAMETER_PATHS["lab_03"]["gateway_role_arn"], role_arn)

print("✅ Gateway deployed with JWT authorization")
print(f"   Gateway ID: {gateway_id}")
print(f"   Gateway URL: {gateway_url}")
print("   Auth Type: Cognito JWT")

## 9. M2M 인증을 사용하여 Runtime을 Gateway 대상으로 추가

  **목표:** OAuth2 M2M 인증을 사용하여 Runtime을 Gateway 대상으로 등록합니다.

  **아키텍처:**

```
  User (JWT) → Gateway (JWT Validation) → Runtime (M2M Token)
                      ↓
              Gateway automatically:
              - Calls GetResourceOauth2Token
              - Retrieves credentials from Secrets Manager
              - Gets M2M access token from Cognito
              - Injects Bearer token in request
                      ↓
              Calls Runtime with Authorization: Bearer {token}
```
  **Gateway가 M2M OAuth2 토큰을 자동으로 전송하는 방식:**

  1. **자격 증명 공급자 저장소**
     - OAuth2 자격 증명 공급자는 `clientId` + `clientSecret`을 AWS Secrets Manager에 저장합니다(암호화됨).
     - 공급자 ARN은 이 보안 저장 위치를 가리킵니다.

  2. **자동 토큰 검색 흐름**
  ```
     Gateway target created with credentialProviderConfigurations
         ↓
     Gateway needs to call Runtime target
         ↓
     Gateway calls GetResourceOauth2Token API (automatic, built-in)
         ↓
     AgentCore Identity retrieves client_id + client_secret from Secrets Manager
         ↓
     AgentCore calls Cognito token endpoint:
     POST /token
     Body: grant_type=client_credentials&client_id=...&client_secret=...&scope=...
         ↓
     Cognito returns M2M access token
         ↓
     AgentCore caches token + manages refresh lifecycle
         ↓
     Gateway injects token in request header:
     Authorization: Bearer {access_token}
         ↓
     Gateway calls Runtime MCP endpoint with Bearer token
         ↓
     Runtime validates JWT signature using Cognito public keys (JWKS)
```
  3. **사용자 지정 코드 불필요**
  - 모든 자격 증명 관리가 자동으로 수행됩니다.
  - 토큰 갱신이 자동으로 수행됩니다.
  - 토큰 주입이 자동으로 수행됩니다.
  - 코드에서는 자격 증명 공급자 ARN만 지정하면 됩니다.

  **핵심 학습 내용:** 이중 인증 - 사용자 기반 수신(JWT), 서비스 기반 송신(OAuth2 M2M).

In [ ]:
# 모든 셀 실행 시나리오를 처리하기 위해 추가
import time

time.sleep(10)

In [ ]:
### 9.1 안전한 액세스를 위한 AgentCore Identity CredentialsProvider 생성
# Lab-01 Cognito 설정에서 M2M 자격 증명 가져오기
m2m_client_id = get_parameter(PARAMETER_PATHS["cognito"]["m2m_client_id"])
m2m_client_secret = get_parameter(PARAMETER_PATHS["cognito"]["m2m_client_secret"])
user_pool_id = get_parameter(PARAMETER_PATHS["cognito"]["user_pool_id"])

# Cognito OIDC discovery URL 구성
discovery_url = f"https://cognito-idp.{AWS_REGION}.amazonaws.com/{user_pool_id}/.well-known/openid-configuration"

# AgentCore 클라이언트 초기화
agentcore = boto3.client("bedrock-agentcore-control", region_name=AWS_REGION)

# OAuth2 Credential Provider 생성
# M2M 자격 증명을 Secrets Manager에 안전하게 저장
credential_provider_response = agentcore.create_oauth2_credential_provider(
    name="aiml301-m2m-credentials",
    credentialProviderVendor="CustomOauth2",
    oauth2ProviderConfigInput={
        "customOauth2ProviderConfig": {
            "clientId": m2m_client_id,
            "clientSecret": m2m_client_secret,
            "oauthDiscovery": {"discoveryUrl": discovery_url},
        }
    },
)

# 응답 추출
oauth2_provider_arn = credential_provider_response["credentialProviderArn"]
client_secret_arn = credential_provider_response["clientSecretArn"]["secretArn"]

# 다음 섹션을 위해 SSM에 저장
put_parameter(PARAMETER_PATHS["lab_03"]["oauth2_provider_arn"], oauth2_provider_arn)
put_parameter(PARAMETER_PATHS["lab_03"]["oauth2_secret_arn"], client_secret_arn)

print("✅ OAuth2 Credential Provider created")
print("\n📋 Credential Storage:")
print(f"   Provider ARN: {oauth2_provider_arn}")
print(f"   Secret ARN: {client_secret_arn}")
print("   Location: AWS Secrets Manager (encrypted)")
print("   Credentials: M2M client_id + client_secret")

In [ ]:
### 9.2 M2M OAuth2를 사용하는 Runtime용 Target 생성

import urllib.parse

# SSM에서 구성 가져오기
gateway_id = get_parameter(PARAMETER_PATHS["lab_03"]["gateway_id"], region_name=AWS_REGION)
runtime_arn = get_parameter(PARAMETER_PATHS["lab_03"]["runtime_arn"], region_name=AWS_REGION)
oauth2_provider_arn = get_parameter(PARAMETER_PATHS["lab_03"]["oauth2_provider_arn"], region_name=AWS_REGION)
resource_server_id = get_parameter(PARAMETER_PATHS["cognito"]["resource_server_identifier"], region_name=AWS_REGION)

# 중요: Runtime ARN으로 엔드포인트 URL 구성
# 형식: https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{URL_ENCODED_ARN}/invocations?qualifier=DEFAULT
# ARN 예시: arn:aws:bedrock-agentcore:us-west-2:123456789012:runtime/my-runtime
encoded_arn = urllib.parse.quote(runtime_arn, safe="")
endpoint_url = (
    f"https://bedrock-agentcore.{AWS_REGION}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT"
)

# 세분화된 권한 부여를 위한 M2M 범위
m2m_scopes = [
    f"{resource_server_id}/mcp.invoke",
    f"{resource_server_id}/runtime.access",
]

# AgentCore 클라이언트 초기화
agentcore = boto3.client("bedrock-agentcore-control", region_name=AWS_REGION)

# Gateway 대상 생성 - M2M OAuth2를 사용하는 외부 MCP 서버로 Runtime 등록
# Gateway는 다음 작업을 자동으로 수행함:
# 1. oauth2_provider_arn을 사용하여 Cognito에서 M2M 토큰 가져오기
# 2. Runtime 호출 시 Bearer 토큰 포함
# 3. Runtime이 토큰을 검증하고 범위에 따라 작업 승인
target_response = agentcore.create_gateway_target(
    gatewayIdentifier=gateway_id,
    name="aiml301-runtime-target",
    description="AgentCore Runtime with M2M OAuth2 authentication",
    targetConfiguration={
        "mcp": {
            "mcpServer": {
                "endpoint": endpoint_url  # ← Runtime ARN을 URL encoding하여 구성
            }
        }
    },
    credentialProviderConfigurations=[
        {
            "credentialProviderType": "OAUTH",
            "credentialProvider": {
                "oauthCredentialProvider": {
                    "providerArn": oauth2_provider_arn,  # ← 9.1절의 OAuth2 credential provider 참조
                    "scopes": m2m_scopes,
                }
            },
        }
    ],
)

target_id = target_response["targetId"]
put_parameter(
    PARAMETER_PATHS["lab_03"]["gateway_runtime_target"],
    target_id,
    region_name=AWS_REGION,
)

print("✅ Runtime added as Gateway target with M2M OAuth2")
print(f"   Target ID: {target_id}")
print(f"   Runtime ARN: {runtime_arn}")
print(f"   Endpoint: {endpoint_url}")
print(f"   Credential Provider: {oauth2_provider_arn}")

In [ ]:
import time

### 9.2 대상 상태 확인 및 동기화

# 대상이 READY 상태가 될 때까지 대기
print("\n⏳ Waiting for target to be READY...")
for attempt in range(30):
    target_info = agentcore.get_gateway_target(gatewayIdentifier=gateway_id, targetId=target_id)
    status = target_info.get("status", "UNKNOWN")

    if status == "READY":
        print("✅ Target is READY")
        break
    if status == "FAILED" or status == "SYNCHRONIZE_UNSUCCESSFUL":
        print(f"❌ Target in ERROR state: {target_info.get('statusReasons', 'No error message')}")
        break
    time.sleep(5)

# 도구 검색을 위해 동기화
agentcore.synchronize_gateway_targets(gatewayIdentifier=gateway_id, targetIdList=[target_id])

print("\n✅ Complete - Gateway will automatically manage M2M tokens")

## 10. JWT 인증을 사용하는 MCP Client 통합

**목표:** Cognito 인증을 통해 로컬 MCP Client를 Gateway에 연결합니다.

**접근 방식:**
1. Cognito에서 토큰 가져오기
2. 로컬 MCP 서버 래퍼 생성
3. MCP 프로토콜을 사용하여 Gateway에 연결
4. 엔드 투 엔드 흐름 테스트

**핵심 학습 내용:** 전체 통합: 로컬 클라이언트 → MCP → Gateway → Runtime → Strands Agent → 문제 해결

In [ ]:
### 10.1: 테스트 사용자 자격 증명 표시
# SSM Parameter Store에서 테스트 사용자 자격 증명 가져오기
user_pool_id = get_parameter(PARAMETER_PATHS["cognito"]["user_pool_id"])
token_endpoint = get_parameter(PARAMETER_PATHS["cognito"]["token_endpoint"])
user_client_id = get_parameter(PARAMETER_PATHS["cognito"]["user_auth_client_id"])
test_username = get_parameter(PARAMETER_PATHS["cognito"]["test_user_email"])
test_password = get_parameter(PARAMETER_PATHS["cognito"]["test_user_password"])

print(f"  ✓ User Pool: {user_pool_id}")
print(f"  ✓ Client ID: {user_client_id}")
print(f"  ✓ Username: {test_username}")

In [ ]:
### 10.2: 테스트 사용자 인증 및 JWT Token 가져오기
print("\n🔑 Authenticating with Cognito...")

cognito = boto3.client("cognito-idp", region_name=AWS_REGION)

response = cognito.initiate_auth(
    ClientId=user_client_id,
    AuthFlow="USER_PASSWORD_AUTH",
    AuthParameters={"USERNAME": test_username, "PASSWORD": test_password},
)

access_token = response["AuthenticationResult"]["AccessToken"]
id_token = response["AuthenticationResult"]["IdToken"]
expires_in = response["AuthenticationResult"]["ExpiresIn"]

print("  ✅ Authentication successful!")
print("  ✓ Token Type: Bearer")
print(f"  ✓ Expires in: {expires_in} seconds")
print(f"  ✓ Access Token (first 50 chars): {access_token[:50]}...")

In [ ]:
### 10.3: JWT 토큰 디코딩 및 클레임 표시

import base64

# 클레임을 표시하기 위해 JWT 토큰 디코딩
print("\n📋 Decoding JWT Token...")

# JWT 토큰은 header.payload.signature의 세 부분으로 구성됨
parts = access_token.split(".")
if len(parts) == 3:
    # payload 디코딩(필요한 경우 패딩 추가)
    payload_b64 = parts[1]
    # base64 디코딩을 위한 패딩 추가
    padding = 4 - len(payload_b64) % 4
    if padding != 4:
        payload_b64 += "=" * padding

payload_json = base64.urlsafe_b64decode(payload_b64)
payload = json.loads(payload_json)

print("  Token Claims:")
print(f"    • Subject (sub): {payload.get('sub', 'N/A')}")
print(f"    • Username: {payload.get('username', 'N/A')}")
print(f"    • Client ID: {payload.get('client_id', 'N/A')}")
print(f"    • Token Use: {payload.get('token_use', 'N/A')}")
print(f"    • Scope: {payload.get('scope', 'N/A')}")
print(f"    • Issued At: {payload.get('iat', 'N/A')}")
print(f"    • Expiration: {payload.get('exp', 'N/A')}")

In [ ]:
### 10.4: MCP Client를 사용하여 AgentCore Gateway에 연결

from lab_helpers.lab_03.mcp_client import MCPClient

print("=" * 80)
print("🌉 Connecting to AgentCore Gateway")
print("=" * 80)

print(f"  ✓ Gateway URL: {gateway_url}")

# MCP Client 생성
client = MCPClient(gateway_url, access_token)
# 세션 초기화
client.initialize()

In [ ]:
### 10.5: Gateway에서 사용 가능한 도구 나열
print("=" * 80)
print("🔧 Step 4: Listing Available MCP Tools")
print("=" * 80)

# Gateway를 통해 사용 가능한 모든 도구 나열
tools = client.list_tools()

# 쉽게 액세스할 수 있도록 도구 이름 저장
tool_names = [tool["name"] for tool in tools]
print(f"\n📝 Available tools: {tool_names}")

In [ ]:
print("=" * 80)
print("🔍 Step 5: Testing Tool Invocation")
print("=" * 80)

# ddgs_search 도구 찾기
search_tool = next(
    (t for t in tools if "infrastructure_agent" in t["name"].lower() and "news" not in t["name"].lower()),
    None,
)

start_time = time.time()
try:
    if search_tool:
        print(f"\n🎯 Using tool: {search_tool['name']}")

        # 검색 도구 호출
        result = client.call_tool(
            tool_name=search_tool["name"],
            arguments={
                "remediation_query": f"""I need help with infrastructure remediation for our CRM application. We're experiencing: {additional_context} """,
                "action_type": "only_plan",
            },
        )

        print("\n✅ End-to-end test complete!")
        print("   Client → Gateway → Runtime → MCP Server → Remediation-Agent ✓")
    else:
        print("❌ Search tool not found")

except Exception as e:
    print(f"❌ Error: {e}")

end_time = time.time()
print(f"  ⏰ Total time taken: {end_time - start_time:.2f} seconds")

## 11. 정리

**목표:** 사용자 지정 Code Interpreter 및 IAM 역할을 포함한 모든 Lab 03 리소스를 제거합니다.

**중요:** 다음에 Lab-04를 실행할 계획이 없는 경우에만 정리를 실행하세요.

In [ ]:
### 11.1: 모든 Lab 03 리소스 정리

from lab_helpers.config import AWS_REGION

# 다음에 Lab-04를 실행할 계획이 없는 경우에만 실행

# cleanup_lab_03(region_name=AWS_REGION, verbose=True)

**다음 단계:**
- ✅ 모든 Lab 03 리소스가 제거되었습니다.
- ✅ 이제 AgentCore Runtime 및 Gateway의 AWS 비용이 발생하지 않습니다.
- ✅ 다른 실습을 실행하거나 Lab 03을 다시 실행할 수 있습니다.

**Lab 03을 다시 실행하려면:**
1. 섹션 1(모듈 가져오기)부터 시작합니다.
2. 모든 헬퍼 함수를 계속 사용할 수 있습니다.
3. IAM 역할이 새로운 권한으로 다시 생성됩니다.

**Lab 04로 이동하려면:**
- 예방 에이전트 워크플로는 `Lab-04-prevention-agent.ipynb`를 참조하세요.

## 요약: Lab 3 - 문제 해결 에이전트 아키텍처

✅ **완료:**
1. ✓ **사용자 지정 Code Interpreter 설정** - IAM 실행 역할을 사용하는 사용자 지정 Interpreter 생성
2. ✓ **사용자 지정 IAM 역할** - CloudWatch, S3, X-Ray 및 Metrics 권한 구성
3. ✓ **PUBLIC 네트워크 모드** - VPC 요구 사항 없이 간소화된 설정
4. ✓ **세션 관리** - 적절한 정리와 함께 Code Interpreter 세션 시작/중지
5. ✓ **문제 해결 도구** - 승인 워크플로를 포함한 계획, 실행 및 검증 도구
6. ✓ **Strands Agent** - 종합적인 시스템 프롬프트를 사용하는 전문 문제 해결 에이전트
7. ✓ **AgentCore Runtime 배포** - 프로덕션 지원 서버리스 배포
8. ✓ **프로덕션 테스트** - 배포된 에이전트의 기능 및 보안 검증
9. ✓ **전체 정리** - 사용자 지정 Interpreter 및 IAM 역할 정리 포함

**전체 워크플로:**
```
Custom Code Interpreter Setup
    ├─ IAM Execution Role ✓
    ├─ Custom Interpreter (PUBLIC mode) ✓
    └─ Session Testing ✓
    ↓
Development (Notebook)
    ↓
Local Testing & Validation
    ↓
AgentCore Runtime Deployment
    ├─ CodeBuild Container Build ✓
    ├─ AWS Resource Creation ✓
    └─ Serverless Deployment ✓
    ↓
Production Agent
    ├─ Serverless Execution ✓
    ├─ Auto-scaling ✓
    ├─ Custom Code Interpreter Integration ✓
    ├─ Approval Workflows ✓
    └─ Monitoring & Logging ✓
    ↓
Cleanup
    ├─ Custom Code Interpreter ✓
    ├─ IAM Execution Role ✓
    └─ All Lab Resources ✓
```
**프로덕션 상태: ✅ 배포 및 운영 중**

**주요 기능:**
- 맞춤형 권한을 사용하는 사용자 지정 Code Interpreter
- 간소화된 설정을 위한 PUBLIC 네트워크 모드
- 실행 환경에 대한 완전한 제어
- 전체 리소스 정리 포함

**다음: Lab 4 - 예방 에이전트** (`Lab-04-prevention-agent.ipynb`)
- AgentCore Browser를 사용한 선제적 인프라 분석
- 실시간 AWS 모범 사례 조사
- 문제가 발생하기 전에 방지하기 위한 예방 중심 권장 사항
- 전체 SRE 자동화 파이프라인: 예방 + 문제 해결